In [30]:
import pandas as pd
import os
import random
from collections import Counter
from typing import Dict, List

# Type alias for clarity: each fold is a dict with three lists of patient IDs (str)
Fold = Dict[str, List[str]]      # {'train': [...], 'val': [...], 'test': [...]}

number_of_folds = 5
validation_fraction = 0.125
# Functions -------------------------
def make_folds_greedy_splits(patient_windows: Dict[str, int],
                              k: int = 5,
                              val_frac: float = 0.10,
                              seed: int = 42) -> List[Fold]:
    """
    Distributes patients across k folds such that each fold has a balanced number of total windows in its test set.
    Then, for each fold, assigns the remaining patients to train and validation sets, trying to preserve window balance.

    Parameters:
    - patient_windows: dict with patient ID (str) as key and number of training windows (int) as value.
    - k: number of folds to generate.
    - val_frac: fraction (0-1) of remaining windows to allocate to validation in each fold.
    - seed: random seed for reproducibility.

    Returns:
    - List of k folds, each a dictionary with 'train', 'val', 'test' patient ID lists.
    """
    rng = random.Random(seed)  # Controlled random generator for reproducibility

    # Convert patient dict to list of (id, window_count) pairs
    items = list(patient_windows.items())
    rng.shuffle(items)  # Shuffle to break ties fairly between patients with equal window counts
    items.sort(key=lambda x: x[1], reverse=True)  # Sort by descending number of windows

    # Step 1: Assign test patients to folds by balancing total number of windows using greedy strategy
    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_load: List[int] = [0] * k  # Keeps track of window total for each fold

    for current_patient_id, current_patient_number_of_windows in items:
        # Find the fold with the smallest current load and assign the patient there
        idx = fold_load.index(min(fold_load))
        test_folds[idx].append(current_patient_id)
        fold_load[idx] += current_patient_number_of_windows

    # Step 2: For each fold, assign the remaining patients to train and val, also attempting to balance window volume
    all_patient_ids = list(patient_windows.keys())
    folds: List[Fold] = []

    for i in range(k):
        test_ids = set(test_folds[i])
        remaining_ids = [patient_id for patient_id in all_patient_ids if patient_id not in test_ids]

        # Sort remaining by descending window count (with shuffle before to break ties randomly)
        rng.shuffle(remaining_ids)
        remaining_items = [(patient_id, patient_windows[patient_id]) for patient_id in remaining_ids]
        remaining_items.sort(key=lambda x: x[1], reverse=True)

        # Target total for validation windows
        total_remaining_windows = sum(count for _, count in remaining_items)
        target_val_windows = val_frac * total_remaining_windows

        val_ids, train_ids = [], []
        val_total, train_total = 0, 0

        # Assign patients to validation until target reached, then assign to train
        for j, (pid, count) in enumerate(remaining_items):
            if val_total < target_val_windows:
                val_ids.append(pid)
                val_total += count
            else:
                break

        train_ids = [pid for pid, _ in remaining_items[j:]]

        # Store the fold split
        folds.append({
            'train': sorted(train_ids),
            'val': sorted(val_ids),
            'test': sorted(test_folds[i])
        })

    return folds

from typing import Dict, List
import random

Fold = Dict[str, List[str]]

def make_folds_greedy_splits_claudia(patient_windows: Dict[str, int],
                              k: int = 5,
                              val_frac: float = 0.10,
                              seed: int = 42) -> List[Fold]:
    """Split patients into *k* folds.

    **Test selection logic (re‑worked)**
    -----------------------------------
    1. **Global ordering** – ALL patients are sorted **once** by their window
       count (descending).  This guarantees that every block of *k* patients
       contains the *k* largest remaining patients.
    2. **Block assignment** – We walk through that ordered list in chunks of
       *k* (e.g. 5 if *k = 5*). Inside each chunk we *shuffle* the IDs and
       assign exactly **one** patient to each test‑fold.  Thus every fold gets
       *one of the top‑k*, *one of the next‑k*, etc.
    3. **Remainder (< k patients)** – If the last block is smaller than *k*,
       those patients are given one‑by‑one to the folds that currently hold
       the fewest **total windows**.

    After test sets are fixed, the remaining patients in each fold are split
    into *train* / *val* using the same greedy balance-by-windows strategy.
    """

    rng = random.Random(seed)

    # ------------------------------------------------------------------
    # 1️⃣  Sort every patient by number of windows (largest → smallest).
    # ------------------------------------------------------------------
    sorted_patients = sorted(patient_windows.keys(),
                             key=lambda pid: patient_windows[pid],
                             reverse=True)

    # ------------------------------------------------------------------
    # 2️⃣  Greedy test‑fold assignment block‑by‑block (size == k).
    # ------------------------------------------------------------------
    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_loads: List[int] = [0] * k  # total windows per test‑fold

    block_size = k  # one patient per fold per block
    n_patients = len(sorted_patients)

    idx = 0
    while idx + block_size <= n_patients:
        block = sorted_patients[idx: idx + block_size]
        rng.shuffle(block)                      # random 1‑to‑1 mapping
        for fold_idx, patient_id in enumerate(block):
            test_folds[fold_idx].append(patient_id)
            fold_loads[fold_idx] += patient_windows[patient_id]
        idx += block_size

    # ------------------------------------------------------------------
    # 3️⃣  Handle leftover patients (< k) → send to lightest fold first.
    # ------------------------------------------------------------------
    for patient_id in sorted_patients[idx:]:
        lightest = fold_loads.index(min(fold_loads))
        test_folds[lightest].append(patient_id)
        fold_loads[lightest] += patient_windows[patient_id]

    # ------------------------------------------------------------------
    # 4️⃣  For each fold: split remaining patients into val / train
    # ------------------------------------------------------------------
    all_ids = set(patient_windows.keys())
    folds: List[Fold] = []

    for fold_idx in range(k):
        test_ids = set(test_folds[fold_idx])
        remaining_ids = list(all_ids - test_ids)  # candidates for train/val

        # Shuffle to break possible ties, then sort by size ↓
        rng.shuffle(remaining_ids)
        remaining_items = [(pid, patient_windows[pid]) for pid in remaining_ids]
        remaining_items.sort(key=lambda x: x[1], reverse=True)

        total_remaining = sum(w for _, w in remaining_items)
        target_val = val_frac * total_remaining

        val_ids, train_ids = [], []
        val_load = 0

        # Fill validation until reaching the target window count
        for j, (pid, windows) in enumerate(remaining_items):
            if val_load < target_val:
                val_ids.append(pid)
                val_load += windows
            else:
                break

        # Whatever is left goes to training
        train_ids.extend(pid for pid, _ in remaining_items[j:])

        folds.append({
            "train": sorted(train_ids),
            "val": sorted(val_ids),
            "test": sorted(test_folds[fold_idx]),
        })

    return folds

def make_folds_greedy_splits_claudia_pure(
    patient_windows: Dict[str, int],
    k: int = 5,
    seed: int = 42,
) -> List[Fold]:
    """Balanced *k*-fold split (patient‑wise).

    • **Test assignment (unchanged)** – patients are sorted globally by window
      count ↓, then taken in *k*-sized blocks; one patient from each block is
      randomly assigned to each test fold so that big patients are spread
      evenly.  Remaining (< *k*) patients go to the fold with the lightest
      load.

    • **Train / Val assignment (NEW)** – the remaining patients of each fold
      are also processed in blocks of *k* **(8 → 7‑train / 1‑val)**.  The last
      block, even if smaller than *k*, still sends **one random patient to
      validation** and the rest to training.  This guarantees *at least one*
      validation patient per fold.
    """

    rng = random.Random(seed)

    # ------------------------------------------------------------------
    # 1️⃣  Sort all patients by window count (descending)
    # ------------------------------------------------------------------
    sorted_patients = sorted(patient_windows.keys(),
                             key=lambda pid: patient_windows[pid],
                             reverse=True)

    # ------------------------------------------------------------------
    # 2️⃣  Assign test patients block‑wise (size == k)
    # ------------------------------------------------------------------
    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_loads: List[int] = [0] * k

    block_size = k  # 5 if k == 5 (one per fold)
    idx = 0
    while idx + block_size <= len(sorted_patients):
        block = sorted_patients[idx: idx + block_size]
        block.sort(key=lambda pid: patient_windows[pid], reverse=True)
        rng.shuffle(block)  # random perm in the block
        for fold_idx, patient_id in enumerate(block):
            test_folds[fold_idx].append(patient_id)
            fold_loads[fold_idx] += patient_windows[patient_id]
        idx += block_size

    # Remaining (< k) patients – assign to lightest fold
    for pid in sorted_patients[idx:]:
        lightest = fold_loads.index(min(fold_loads))
        test_folds[lightest].append(pid)
        fold_loads[lightest] += patient_windows[pid]

    # ------------------------------------------------------------------
    # 3️⃣  Build train / val with 7‑train / 1‑val blocks (size == 8)
    # ------------------------------------------------------------------
    all_ids_set = set(patient_windows.keys())
    folds: List[Fold] = []
    train_block = 7  # patients for train per block of 8
    val_block = 1    # patients for val  per block of 8
    block_tv = train_block + val_block  # 8

    for fold_idx in range(k):
        test_ids = set(test_folds[fold_idx])
        remaining_ids = [pid for pid in sorted_patients if pid not in test_ids]

        # --- process in blocks of 8 ------------------------------------
        train_ids, val_ids = [], []
        ptr = 0
        n_remaining = len(remaining_ids)

        while ptr + block_tv <= n_remaining:
            block = remaining_ids[ptr: ptr + block_tv]
            # already sorted globally; still shuffle to randomise who goes to val
            rng.shuffle(block)
            val_ids.append(block[0])          # 1 patient → val
            train_ids.extend(block[1:])       # 7 patients → train
            ptr += block_tv

        # Handle last (incomplete) block → send 1 to val, rest to train
        tail = remaining_ids[ptr:]
        if tail:
            rng.shuffle(tail)
            val_ids.append(tail[0])
            train_ids.extend(tail[1:])

        folds.append({
            'train': sorted(train_ids),
            'val': sorted(val_ids),
            'test': sorted(test_folds[fold_idx])
        })

    return folds

def validate_folds(folds: List[Fold], patient_measurements: Dict[str, int], tolerance_pct: float = 10.0) -> None:    
    """
    Validate a list of patient-wise cross-validation folds.

    Parameters
    ----------
    folds : list[Fold]
        Output of `make_folds_greedy_splits` or similar.  Each element is a
        dictionary with keys ``'train'``, ``'val'``, ``'test'`` mapping to lists
        of patient IDs **(strings)**.

    patient_measurements : dict[str, int]
        Mapping ``{patient_id: number_of_measurements}``.  Used to check load
        balancing (Rule R4).

    tolerance_pct : float, default 10.0
        Maximum allowed percentage deviation from the mean test load before
        a warning is raised (Rule R4).
    
    horizon : int, default 2
        The horizon used for the folds, included in the printed messages.

    Raises
    ------
    AssertionError
        If any of the mandatory rules (R1–R3) is violated.

    Side effects
    ------------
    * Prints a success message for each rule that passes.
    * Prints the test-set load per fold and the percentage deviation from the
      mean (always), and a warning if any fold exceeds *tolerance_pct*.

    Notes
    -----
    *R4* (load balance) is **informative**; it does **not** raise an error.  
    Adjust *tolerance_pct* as appropriate for your project.  
    """
    
    print(f"\nValidating folds ...")
    
    all_patients = set(patient_measurements)

    # ------------------------------------------------------------------ R1/R3
    for idx, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        # R1 – exclusivity within the same fold
        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), (
            f"[R1] Overlap detected in fold {idx}"
        )

        # R3 – union covers the entire cohort
        assert len(tr | va | te) == len(all_patients), (
            f"[R3] Missing patients in fold {idx}"
        )

    print("\n ✓ R1 and R3 passed for every fold.")

    # ------------------------------------------------------------------ R2
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, cnt in test_counter.items() if cnt != 1]

    assert not duplicates, (
        f"[R2] Patients appearing in test 0 or >1 times: {duplicates}"
    )
    print("✓ R2 passed – each patient appears in *test* exactly once.")

    # ------------------------------------------------------------------ R4 (informative)
    def test_load(ids: List[str]) -> int:
        """Total number of measurements for this group of patients."""
        return sum(patient_measurements[pid] for pid in ids)

    loads = [test_load(f['test']) for f in folds]
    mean_load = sum(loads) / len(loads)
    pct_dev = [100 * (l - mean_load) / mean_load for l in loads]

    print("\nTest-set load per fold (number of measurements):", loads)
    print("Deviation from mean (%)                      :", [round(p, 2) for p in pct_dev])

    if any(abs(p) > tolerance_pct for p in pct_dev):
        print(f"⚠️  [R4] Warning: at least one fold deviates by more than {tolerance_pct}% "
              "from the mean test load.")
    else:
        print(f"✓ R4 passed – test loads within ±{tolerance_pct}% of the mean.")

    print(f"\nAll mandatory validation checks completed successfully.")
    
def tag_folds_and_save(
            df_windows: pd.DataFrame,
            folds: List[Fold],
            output_path: str,
            filename: str = "windows_with_folds_horizon_{horizon}.parquet",
            horizon: int = 2,
            column_prefix: str = "fold") -> None:
    
    """
    Add one column per fold (fold0 … fold{k-1}) indicating whether each window
    belongs to 'train', 'val' or 'test' in that fold, then write a single
    Parquet file.

    Parameters
    ----------
    df_windows : pd.DataFrame
        Original window table (must include column 'patient_id').

    folds : list[Fold]
        Output from make_folds_greedy_splits (patient IDs per split).

    output_path : str
        Destination filename, e.g. 'windows_with_folds.parquet'.

    column_prefix : str, default "fold"
        Prefix for the new columns: fold0, fold1, …

    parquet_kwargs : dict, optional
        Extra arguments passed to DataFrame.to_parquet.
    """    
    df_tagged = df_windows.copy()

    for idx, fold in enumerate(folds):
        # Build a mapping patient_id -> split label for this fold
        label_map = {pid: "train" for pid in fold["train"]}
        label_map.update({pid: "val"  for pid in fold["val"]})
        label_map.update({pid: "test" for pid in fold["test"]})

        col_name = f"{column_prefix}{idx}"
        df_tagged[col_name] = df_tagged["patient_id"].map(label_map).astype("category")

    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    # df_tagged.to_parquet(f'{output_path}{filename}_horizon_{horizon}.parquet', index=False)
    print(f"\n ✓ File {filename}_horizon_{horizon} written: {output_path}  "
          f"({len(df_windows):,} rows, {len(folds)} fold-columns)")
    
    return df_tagged


# Count the number of windows per patient and convert to a dictionary
patient_number_windows_dict = {
    'P1': 1000,
    'P2': 950,
    'P3': 900,
    'P4': 850,
    'P5': 800,
    'P6': 750,
    'P7': 700,
    'P8': 100,
    'P9': 100,
    'P10': 100,
    'P11': 650,
    'P12': 600,
    'P13': 550,
    'P14': 500,
    'P15': 450,
    'P16': 400,
    'P17': 350,
    'P18': 100,
    'P19': 100,
    'P20': 100,
}

num_patients = len(patient_number_windows_dict)

# Create folds using the greedy split function
folds_split = make_folds_greedy_splits(patient_number_windows_dict, k=number_of_folds, val_frac=validation_fraction, seed=42)


# Print statistics of the folds created
print(f'\n ------- Statistics of folds created -------')
for i, fold in enumerate(folds_split):
    print(f"\n---- Fold {i + 1}: -----")
    print("-- Totals --")
    print(f"  Train: {len(fold['train'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['train'])}")
    print(f"  Val:   {len(fold['val'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['val'])}") 
    print(f"  Test:  {len(fold['test'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['test'])}")
    
    print("\n -- Percentages --")
    print(f"  Train: {(len(fold['train'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['train']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Val:   {(len(fold['val'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['val']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Test:  {(len(fold['test'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['test']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")

print(f'\nEnd of statistics for folds created ')

# Validate the folds created
validate_folds(folds=folds_split, patient_measurements=patient_number_windows_dict)

# save_folds_as_parquet(df_windows, folds_split, "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/")


 ------- Statistics of folds created -------

---- Fold 1: -----
-- Totals --
  Train: 15 patients. Total windows: 6150
  Val:   2 patients. Total windows: 1850
  Test:  3 patients. Total windows: 2050

 -- Percentages --
  Train: 75.00 percentage of patients. Percentage of windows: 61.19%
  Val:   10.00 percentage of patients. Percentage of windows: 18.41%
  Test:  15.00 percentage of patients. Percentage of windows: 20.40%

---- Fold 2: -----
-- Totals --
  Train: 15 patients. Total windows: 6150
  Val:   2 patients. Total windows: 1900
  Test:  3 patients. Total windows: 2000

 -- Percentages --
  Train: 75.00 percentage of patients. Percentage of windows: 61.19%
  Val:   10.00 percentage of patients. Percentage of windows: 18.91%
  Test:  15.00 percentage of patients. Percentage of windows: 19.90%

---- Fold 3: -----
-- Totals --
  Train: 15 patients. Total windows: 7000
  Val:   1 patients. Total windows: 1000
  Test:  4 patients. Total windows: 2050

 -- Percentages --
  Train: 

In [28]:

# Save the folds as Parquet files

folds_split_claudia = make_folds_greedy_splits_claudia(patient_number_windows_dict, k=number_of_folds, val_frac=validation_fraction, seed=42)


# Print statistics of the folds created
print(f'\n ------- Statistics of folds created -------')
for i, fold in enumerate(folds_split_claudia):
    print(f"\n---- Fold {i + 1}: -----")
    print("-- Totals --")
    print(f"  Train: {len(fold['train'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['train'])}")
    print(f"  Val:   {len(fold['val'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['val'])}") 
    print(f"  Test:  {len(fold['test'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['test'])}")
    
    print("\n -- Percentages --")
    print(f"  Train: {(len(fold['train'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['train']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Val:   {(len(fold['val'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['val']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Test:  {(len(fold['test'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['test']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")

print(f'\nEnd of statistics for folds created ')

# Validate the folds created
validate_folds(folds=folds_split_claudia, patient_measurements=patient_number_windows_dict)


 ------- Statistics of folds created -------

---- Fold 1: -----
-- Totals --
  Train: 14 patients. Total windows: 6200
  Val:   2 patients. Total windows: 1950
  Test:  4 patients. Total windows: 1900

 -- Percentages --
  Train: 70.00 percentage of patients. Percentage of windows: 61.69%
  Val:   10.00 percentage of patients. Percentage of windows: 19.40%
  Test:  20.00 percentage of patients. Percentage of windows: 18.91%

---- Fold 2: -----
-- Totals --
  Train: 14 patients. Total windows: 6000
  Val:   2 patients. Total windows: 1900
  Test:  4 patients. Total windows: 2150

 -- Percentages --
  Train: 70.00 percentage of patients. Percentage of windows: 59.70%
  Val:   10.00 percentage of patients. Percentage of windows: 18.91%
  Test:  20.00 percentage of patients. Percentage of windows: 21.39%

---- Fold 3: -----
-- Totals --
  Train: 14 patients. Total windows: 5950
  Val:   2 patients. Total windows: 1950
  Test:  4 patients. Total windows: 2150

 -- Percentages --
  Train: 

In [29]:

# Save the folds as Parquet files

folds_split_claudia_pure = make_folds_greedy_splits_claudia_pure(patient_number_windows_dict, k=number_of_folds, seed=42)


# Print statistics of the folds created
print(f'\n ------- Statistics of folds created -------')
for i, fold in enumerate(folds_split_claudia):
    print(f"\n---- Fold {i + 1}: -----")
    print("-- Totals --")
    print(f"  Train: {len(fold['train'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['train'])}")
    print(f"  Val:   {len(fold['val'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['val'])}") 
    print(f"  Test:  {len(fold['test'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['test'])}")
    
    print("\n -- Percentages --")
    print(f"  Train: {(len(fold['train'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['train']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Val:   {(len(fold['val'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['val']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Test:  {(len(fold['test'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['test']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")

print(f'\nEnd of statistics for folds created ')

# Validate the folds created
validate_folds(folds=folds_split_claudia_pure, patient_measurements=patient_number_windows_dict)


 ------- Statistics of folds created -------

---- Fold 1: -----
-- Totals --
  Train: 14 patients. Total windows: 6200
  Val:   2 patients. Total windows: 1950
  Test:  4 patients. Total windows: 1900

 -- Percentages --
  Train: 70.00 percentage of patients. Percentage of windows: 61.69%
  Val:   10.00 percentage of patients. Percentage of windows: 19.40%
  Test:  20.00 percentage of patients. Percentage of windows: 18.91%

---- Fold 2: -----
-- Totals --
  Train: 14 patients. Total windows: 6000
  Val:   2 patients. Total windows: 1900
  Test:  4 patients. Total windows: 2150

 -- Percentages --
  Train: 70.00 percentage of patients. Percentage of windows: 59.70%
  Val:   10.00 percentage of patients. Percentage of windows: 18.91%
  Test:  20.00 percentage of patients. Percentage of windows: 21.39%

---- Fold 3: -----
-- Totals --
  Train: 14 patients. Total windows: 5950
  Val:   2 patients. Total windows: 1950
  Test:  4 patients. Total windows: 2150

 -- Percentages --
  Train: 

In [19]:
for fold in folds_split_claudia:
    print(fold['test'])


['P10', 'P12', 'P17', 'P4']
['P11', 'P15', 'P18', 'P2']
['P16', 'P19', 'P3', 'P6']
['P13', 'P14', 'P20', 'P5']
['P1', 'P7', 'P8', 'P9']


In [20]:
for fold in folds_split_claudia:
    print(fold['val'])

['P1', 'P2']
['P1', 'P3']
['P1', 'P2']
['P1', 'P2']
['P2', 'P3']


In [21]:
for fold in folds_split_claudia:
    print(fold['train'])

['P11', 'P13', 'P14', 'P15', 'P16', 'P18', 'P19', 'P20', 'P3', 'P5', 'P6', 'P7', 'P8', 'P9']
['P10', 'P12', 'P13', 'P14', 'P16', 'P17', 'P19', 'P20', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9']
['P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P17', 'P18', 'P20', 'P4', 'P5', 'P7', 'P8', 'P9']
['P10', 'P11', 'P12', 'P15', 'P16', 'P17', 'P18', 'P19', 'P3', 'P4', 'P6', 'P7', 'P8', 'P9']
['P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P4', 'P5', 'P6']
